In [3]:
import os, sys
import pyspark

os.environ["PYSPARK_PYTHON"]=sys.executable          # Worker가 사용할 Python 실행 파일 경로 설정 (리눅스 "/usr/bin/python3")
os.environ["PYSPARK_DRIVER_PYTHON"]=sys.executable   # Driver에서도 동일한 Python 경로 설정
# os.environ['HADOOP_HOME']=os.getcwd() # 현재 디렉터리를 HADOOP_HOME으로 설정
# os.environ["PATH"] += os.path.join(os.environ['HADOOP_HOME'], 'bin') # PATH에 Hadoop 바이너리 추가

myConf=pyspark.SparkConf() # 기본 설정 객체 생성, 여기에 필요한 설정 정의
# myConf=pyspark.SparkConf().set("spark.driver.bindAddress", "127.0.0.1") #드라이버 바인딩 주소 설정
# myConf=pyspark.SparkConf().set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.1.1") 

In [4]:
spark = pyspark.sql.SparkSession\
    .builder\
    .master("local")\
    .appName("test")\
    .config(conf=myConf)\
    .getOrCreate()

In [5]:
_ohlcv = [

    ("2024-01-03", 100.0, 110.0, 95.0, 105.0, 10000.0),

    ("2024-01-04", 105.0, 115.0, 100.0, 110.0, 12000.0),

    ("2024-01-05", 110.0, 120.0, 105.0, 115.0, 15000.0),

    ("2024-01-08", 105.0, 115.0, 105.0, 110.0, 11000.0),

    ("2024-01-09", 115.0, 120.0, 105.0, 115.0, 17000.0)

]

In [6]:
jsDf = spark.createDataFrame([(_ohlcv[i][0], _ohlcv[i][1:]) for i in range(5)],\
                           ["year-month-day","ohlcv"])

In [8]:
from pyspark.sql.types import *

# 스키마 정의
schema = StructType([
    StructField("year-month-day", StringType(), True),  # 날짜
    StructField("ohlcv", ArrayType(FloatType()), True)  # OHLCV 데이터
])

# 데이터프레임 생성
js = spark.createDataFrame(data=_ohlcv, schema=schema)

# 스키마 출력
js.printSchema()

# 데이터 출력
js.show()

PySparkValueError: [LENGTH_SHOULD_BE_THE_SAME] obj and fields should be of the same length, got 6 and 2.

In [ ]:
jsDf.printSchema()

In [ ]:
jsDf.show(10)

In [ ]:
closing_prices = [row["ohlcv"][3] for row in jsDf.collect()]

print(closing_prices)

In [ ]:
# Python으로 기본 통계 계산
count = len(closing_prices)
mean = sum(closing_prices) / count
minimum = min(closing_prices)
maximum = max(closing_prices)

# 출력
print(f"Count: {count}")
print(f"Mean: {mean}")
print(f"Min: {minimum}")
print(f"Max: {maximum}")

In [ ]:
#plus 추가로 찾은 방법 => Series 활용

import pandas as pd


closing_prices = [row['ohlcv'][3] for row in jsDf.collect()]
print(pd.Series(closing_prices).describe())

In [ ]:
from pyspark.sql.functions import split

#리스트 안에 문자열이 들어왔네 => dict형태로 참조하였군
split_col = split(jsDf['year-month-day'], '-')

In [ ]:
jsDf = jsDf.withColumn('year', split_col.getItem(0))

In [ ]:
jsDf.show()

In [ ]:
jsDf.select("year").show()